# Clustering

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

### Setup

In [ ]:
import os
from pathlib import Path
import scanpy as sc
import numpy as np
from scipy.sparse import issparse, csr_matrix
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import warnings
import session_info
import sys

In [ ]:
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning) # Suppress Pandas PerformanceWarnings related to fragmentation

plt.rcParams['figure.figsize'] = (3, 3)

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR

input_dir = os.path.join(BASE_DIR, 'data/h5ad/export_02/02a_scvi')
clusters_dir = os.path.join(BASE_DIR, 'clusters')
output_dir = os.path.join(BASE_DIR, 'data/h5ad/export_02/02b_leiden')

os.makedirs(clusters_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
adata = sc.read_h5ad(os.path.join(input_dir, 'ModelA_475_adata-scvi.h5ad'))
bdata = sc.read_h5ad(os.path.join(input_dir, 'ModelB_480_adata-scvi.h5ad'))

adata

## Prepare data

Importing as raw counts layer. Use log1p for rank genes.

In [ ]:
print(adata.X[:5, :5].toarray())
print('')
print('adata.X has only whole numbers:', np.all(adata.X.data == np.round(adata.X.data)))  # True if all values are whole numbers

In [ ]:
adata.layers

In [ ]:
# Set to log1p counts layer
adata.X = adata.layers['log1p'].copy()

In [ ]:
print(adata.X[:5, :5].toarray())
print('')
print('adata.X has only whole numbers:', np.all(adata.X.data == np.round(adata.X.data)))

## Neighbors, UMAP, clustering

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI_ModelA_475', random_state = 0)
sc.pp.neighbors(bdata, use_rep = 'X_scVI_ModelB_480', random_state = 0)

In [ ]:
sc.tl.umap(adata, random_state = 0)
sc.tl.umap(bdata, random_state = 0)

In [ ]:
sc.pl.umap(adata)
sc.pl.umap(bdata)

In [ ]:
sc.tl.leiden(adata, key_added='leiden_scVI_1', resolution=1)
sc.tl.leiden(bdata, key_added='leiden_scVI_1', resolution=1)

In [ ]:
for data in [adata, bdata]:
    sc.pl.umap(data, color=['leiden_scVI_1'], legend_fontsize=10, frameon=False)
    sc.pl.umap(data, color=['sample_id'],     legend_fontsize=10, frameon=False)
    sc.pl.umap(data, color=['slide_id'],         legend_fontsize=10, frameon=False)

### Run DE rankings on clusters

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_scVI_1', method='wilcoxon')
df = sc.get.rank_genes_groups_df(adata, group = None)
df = df.copy()
df = df[(df.pvals_adj < 0.05) & (df.logfoldchanges > .5)].copy()
df.head(5)

In [ ]:
clusters_path = os.path.join(clusters_dir, 'rank-genes-clusters.csv')

df.to_csv(clusters_path, index=False)
print(clusters_path)

In [ ]:
# Visualize top results
sc.pl.rank_genes_groups(adata, groupby = 'leiden_scVI', method = 'wilcoxon')

In [ ]:
adata_path = os.path.join(output_dir, 'adata-leiden-475.h5ad')
bdata_path = os.path.join(output_dir, 'adata-leiden-480.h5ad')

adata.write_h5ad(adata_path, compression='gzip')
bdata.write_h5ad(bdata_path, compression='gzip')

print(adata_path)
print(bdata_path)